# 01 · Understand `uk_address_matcher` end-to-end

`uk_address_matcher` is a **geocoder / address matcher**: given a *messy* address it
finds the best-matching *canonical* address (typically Ordnance Survey AddressBase or
NGD). It runs on a laptop, is built on **DuckDB** (SQL engine) + **Splink**
(probabilistic record linkage), and ships a pre-trained Splink model.

This notebook answers: *what does it actually do, and what are the moving parts?*
Later notebooks dig into two concrete contribution opportunities:

- **02** — adding abbreviation expansions (`RC → ROMAN CATHOLIC`, `HMP → HIS MAJESTYS PRISON`).
- **03** — keeping car-park "addresses" out of the canonical candidate pool.

> First run downloads two small parquet datasets from GitHub (cached afterwards).

## Inputs

Both the messy and canonical tables need just two columns (a third is recommended):

| Column | Type | Required | Purpose |
|---|---|---|---|
| `unique_id` | BIGINT or VARCHAR | yes | stable per-row id |
| `address_concat` | VARCHAR | yes | the address text (ideally **excluding** postcode) |
| `postcode` | VARCHAR | recommended | used for blocking + matching; parsed out of `address_concat` if absent |

In [1]:
import duckdb
from uk_address_matcher import (
    AddressMatcher, ExactMatchStage, PeeledAddressStage, SplinkStage, ukam_datasets,
)

con = duckdb.connect(database=":memory:")

# Bundled fictional dataset (downloads on first use, then cached).
messy, canonical = ukam_datasets.fictional_london
print("messy columns   :", messy.columns)
print("canonical columns:", canonical.columns)
print("messy rows      :", messy.aggregate("count(*) AS n").fetchone()[0])
print("canonical rows  :", canonical.aggregate("count(*) AS n").fetchone()[0])

messy columns   : ['unique_id', 'address_concat', 'postcode', 'ukam_label']
canonical columns: ['unique_id', 'address_concat', 'postcode']
messy rows      : 2000
canonical rows  : 10000


In [3]:
import duckdb
from uk_address_matcher import AddressMatcher, ukam_datasets

con = duckdb.connect()

df_messy = ukam_datasets.as_relation("fictional_london_messy", con=con)
df_canonical = ukam_datasets.as_relation("fictional_london_canonical", con=con)

matcher = AddressMatcher(
    canonical_addresses=df_canonical,
    addresses_to_match=df_messy,
    con=con,
)
result = matcher.match()
print(result.matches().limit(5).to_df().to_string(index=False))

unique_id resolved_canonical_id ukam_label                             original_address_concat                      original_address_concat_canonical                match_reason  match_weight  distinguishability
m_0001872             c_0005356  c_0005356        FLAT 2,131 PRIMROSEWICK WY,WEST ALDER,LONDON       Flat 2, 131 Primrosewick Way, West Alder, London           exact: full match           NaN                 NaN
m_0001832             c_0000916  c_0000916 Unit 6, 102 Bartonstone Gdns, East Bramwick, London Unit 6, 102 Bartonstone Gardens, East Bramwick, London           exact: full match           NaN                 NaN
m_0001440             c_0008498  c_0008498              Suite 9,3 Elmhurst St,Kingsford,London          Suite 9, 3 Elmhurst Street, Kingsford, London splink: probabilistic match     40.030048                 NaN
m_0001762             c_0003898  c_0003898       UNIT 11, 12 YORKSTNOE ST, MAPLE GREEN, LONDON      Unit 11, 12 Yorkstone Street, Maple Green, London sp

In [ ]:

import duckdb
import os
import tempfile
from uk_address_matcher import (
    AddressMatcher,
    prepare_canonical_folder,
    ukam_datasets,
)

con = duckdb.connect()
df_messy = ukam_datasets.as_relation("fictional_london_messy", con=con)
df_canonical = ukam_datasets.as_relation("fictional_london_canonical", con=con)

# One-time preparation
output_folder = tempfile.mkdtemp()
prepare_canonical_folder(
    df_canonical,
    output_folder=output_folder,
    con=con,
    overwrite=True,
)

# Pass the folder path instead of a relation
matcher = AddressMatcher(
    canonical_addresses=output_folder,
    addresses_to_match=df_messy,
    con=con,
)
result = matcher.match()

print("Prepared folder contents:")
for f in sorted(os.listdir(output_folder)):
    print(f"  {f}")
print()
print(result.matches().limit(5).to_df().to_markdown(index=False))

Prepared folder contents:
  ukam_canonical_addresses.parquet
  ukam_inverted_index.parquet
  ukam_manifest.json
  ukam_term_frequencies.parquet

unique_id resolved_canonical_id ukam_label                             original_address_concat                      original_address_concat_canonical                match_reason  match_weight  distinguishability
m_0001872             c_0005356  c_0005356        FLAT 2,131 PRIMROSEWICK WY,WEST ALDER,LONDON       Flat 2, 131 Primrosewick Way, West Alder, London           exact: full match           NaN                 NaN
m_0001832             c_0000916  c_0000916 Unit 6, 102 Bartonstone Gdns, East Bramwick, London Unit 6, 102 Bartonstone Gardens, East Bramwick, London           exact: full match           NaN                 NaN
m_0001440             c_0008498  c_0008498              Suite 9,3 Elmhurst St,Kingsford,London          Suite 9, 3 Elmhurst Street, Kingsford, London splink: probabilistic match     40.030048                 NaN
m_00017

In [2]:
# Use .df() throughout — it renders as a clean table in Jupyter and avoids
# Windows-console encoding issues you'd hit with relation.show().
messy.limit(10).df()

,unique_id,address_concat,postcode,ukam_label
0,m_0000001,"91 Oakmead Close,Kingsford,London",N2 1VW,c_0008999
1,m_0000002,"93 LAURELPOINT GDNS,KINGSFORD,LONDON",SE12 2HR,c_0001345
2,m_0000003,"202 Laurelmead St,New Huxley,London",EC123AT,c_0003647
3,m_0000004,"49 Novabank Ave,Kingsford,London",SW3 4YW,c_0004292
4,m_0000005,"74 GROVESTEAD LN, MAPLE GREEN, LONDON",SE11YJ,c_0002543
5,m_0000006,"50 QUARRYWICK CL,MAPLE GREEN,LONDON",SW3 9PB,c_0000395
6,m_0000007,"FLAT 19,228 ZEPHYRGATE ROAD,EAST BRAMWICK,LONDON",WC20 8BZ,c_0005719
7,m_0000008,"167 Novawood Gdns, Rivermill, London",SW8 0VZ,c_0003344
8,m_0000009,"Suite 51, 128 Cliftonpoint St, New Huxley, London",SE15 3CW,c_0009899
9,m_0000010,"Apt 76,55 Grovemill Rd,Kingsford,London",N128SA,c_0005700


In [4]:
canonical.limit(5).df()

,unique_id,address_concat,postcode
0,c_0000001,"233 Hawmill Lane, West Alder, London",EC5 7GD
1,c_0000002,"Flat 7, 68 Unionheath Lane, South Norland, London",E8 8UT
2,c_0000003,"36 Marlowmarket Road, South Norland, London",NW10 2DV
3,c_0000004,"96 Marlowhill Street, Kingsford, London",NW24 2CW
4,c_0000005,"130 Hawview Road, West Alder, London",SE7 5RH


## The matching pipeline = a list of **stages**

`AddressMatcher` runs an ordered list of stages. Earlier stages settle the easy cases
cheaply; the probabilistic stage handles the fuzzy remainder.

- `ExactMatchStage()` — deterministic / exact matches.
- `PeeledAddressStage()` — handles "peeling" leading sub-building tokens.
- `UniqueTrigramStage()` — match on a canonical-unique trigram.
- `SplinkStage(...)` — probabilistic matching (typos, reordering, missing tokens),
  using the pre-trained Splink model and **term-frequency** weighting.

Behind the scenes the canonical side is cleaned, term frequencies are computed, and an
**inverted index** (trigram/bigram keys → canonical ids) builds each messy record's
candidate pool (`exploding_unique_ids`) so Splink only scores plausible pairs.

In [5]:
print(AddressMatcher.available_stages())

Available matching stages (import from uk_address_matcher):
  ExactMatchStage    — Deterministic exact matching on ``clean_full_address`` and ``postcode``.
  PeeledAddressStage — Deterministic matching after peeling common UK locality suffixes.
  SplinkStage        — Probabilistic matching stage built on Splink.
  UniqueTrigramStage — Deterministic matching using n-grams that identify one canonical row.

Usage:
  from uk_address_matcher import ExactMatchStage, PeeledAddressStage, SplinkStage, UniqueTrigramStage


In [6]:
matcher = AddressMatcher(
    canonical_addresses=canonical,
    addresses_to_match=messy,
    con=con,
    stages=[
        ExactMatchStage(),
        PeeledAddressStage(),
        SplinkStage(
            predict_threshold_match_weight=-20,
            final_match_weight_threshold=12,
            include_full_postcode_block=True,
            retain_intermediate_calculation_columns=True,
        ),
    ],
)
result = matcher.match()
result.matches().limit(10).df()

,unique_id,resolved_canonical_id,ukam_label,original_address_concat,original_address_concat_canonical,match_reason,match_weight,distinguishability
0,m_0001872,c_0005356,c_0005356,"FLAT 2,131 PRIMROSEWICK WY,WEST ALDER,LONDON","Flat 2, 131 Primrosewick Way, West Alder, London",exact: full match,NaN,NaN
1,m_0001832,c_0000916,c_0000916,"Unit 6, 102 Bartonstone Gdns, East Bramwick, L...","Unit 6, 102 Bartonstone Gardens, East Bramwick...",exact: full match,NaN,NaN
2,m_0001440,c_0008498,c_0008498,"Suite 9,3 Elmhurst St,Kingsford,London","Suite 9, 3 Elmhurst Street, Kingsford, London",splink: probabilistic match,40.030048,NaN
3,m_0001762,c_0003898,c_0003898,"UNIT 11, 12 YORKSTNOE ST, MAPLE GREEN, LONDON","Unit 11, 12 Yorkstone Street, Maple Green, London",splink: probabilistic match,38.044863,NaN
4,m_0001689,c_0007665,c_0007665,"Suite 12, 120 Novalane Ave, New Huxley, London","Suite 12, 120 Novalane Avenue, New Huxley, London",exact: full match,NaN,NaN
5,m_0001881,c_0009241,c_0009241,"Suite 14, 206 Quarrystead Rd, New Huxely, London","Suite 14, 206 Quarrystead Road, New Huxley, Lo...",splink: probabilistic match,45.687276,NaN
6,m_0001792,c_0003921,c_0003921,"Apt 15,170 Falconmill Ave,North Gresytone,London","Apt 15, 170 Falconmill Avenue, North Greystone...",splink: probabilistic match,45.671812,NaN
7,m_0001408,c_0005878,c_0005878,"APT 17, 122 AMBERHURST LN, NEW HUXLEY, LONDON","Apt 17, 122 Amberhurst Lane, New Huxley, London",exact: full match,NaN,NaN
8,m_0001487,c_0005470,c_0005470,"Apt 18,247 Elmheath St,Rivermill,London","Apt 18, 247 Elmheath Street, Rivermill, London",splink: probabilistic match,40.836225,NaN
9,m_0001364,c_0005719,c_0005719,"FLAT 19,228 ZEPHYRGATE RD,EAST BRAMWICK,LONDON","Flat 19, 228 Zephyrgate Road, East Bramwick, L...",exact: full match,NaN,NaN


### Output columns

| Column | Meaning |
|---|---|
| `unique_id` | the messy record |
| `resolved_canonical_id` | the chosen canonical `unique_id` |
| `original_address_concat` / `..._canonical` | the two address texts |
| `match_reason` | which stage decided (e.g. `splink: probabilistic match`) |
| `match_weight` | Splink confidence (log2 Bayes factor; higher = more confident) |
| `distinguishability` | gap in weight to the next-best candidate (bigger = less ambiguous) |

In [7]:
# How many matches came from each stage / reason?
result.match_metrics().df()

,match_reason,match_count,match_percentage
0,exact: full match,1182,59.10%
1,splink: probabilistic match,718,35.90%
2,unmatched,100,5.00%


## Peeking under the hood: the candidate pool

`._splink_predictions()` returns every *candidate pair* Splink scored — not just the
winner. This is the lens we use in notebook 03 to see bad candidates (car parks)
sitting in the pool next to the right answer.

In [7]:
sp = result._splink_predictions(limit=5)
# A few of the most useful columns:
cols = [c for c in ["match_weight", "match_probability",
                    "original_address_concat_l", "original_address_concat_r"]
        if c in sp.columns]
sp.select(", ".join(cols)).order("match_weight DESC").df()

,match_weight,match_probability,original_address_concat_l,original_address_concat_r
0,12.298789,0.999802,"58 Yorkbridge Street, South Norland, London","58 Yorkbridge St, South Norland, London"
1,11.517204,0.999659,"160 Marlowhill Street, Canal Quarter, London","160 MARLOWHILL ST, CANAL QUARTER, LONDON"
2,10.522611,0.999321,"127 Juniperstead Close, South Norland, London","127 Juniprestead Close, South Norland, London"
3,8.606356,0.997441,"207 Kestrelmill Street, West Alder, London","207 Kesrtelmill St,West Alder,London"
4,8.256404,0.996740,"42 Bartonview Street, South Norland, London","42 BATRONVIEW ST,SOUTH NORLAND,LONDON"


## Takeaways & where it can be improved

- The matcher is a **staged pipeline** over DuckDB + a pre-trained Splink model.
- **Cleaning** (normalisation, abbreviation expansion) happens before matching and
  strongly affects which candidates look similar → **notebook 02**.
- The **canonical candidate pool** can contain records that aren't real dwellings
  (e.g. car-park spaces) which then compete with the right answer → **notebook 03**.